In [1]:
import requests
import torch
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor, AutoTokenizer
# some notes from SAE tutorial on gemmascope https://colab.research.google.com/drive/17dQFYUYnuKnP6OwQPH9v_GSYUW5aj-Rp#scrollTo=12wF3f7o1Ni7

torch.set_grad_enabled(False) 

/Users/enjalot/code/touch-tokens/ttenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = "meta-llama/Llama-3.2-11B-Vision"

In [3]:

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    # torch_dtype=torch.bfloat16,
    # device_map="auto",
    device_map="cpu",
)

Loading checkpoint shards: 100%|██████████| 5/5 [00:31<00:00,  6.25s/it]


In [4]:
print(model)

MllamaForConditionalGeneration(
  (vision_model): MllamaVisionModel(
    (patch_embedding): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), padding=valid, bias=False)
    (gated_positional_embedding): MllamaPrecomputedPositionEmbedding(
      (tile_embedding): Embedding(9, 5248000)
    )
    (pre_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
      (embedding): Embedding(9, 5120)
    )
    (post_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
      (embedding): Embedding(9, 5120)
    )
    (layernorm_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (layernorm_post): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): MllamaVisionEncoder(
      (layers): ModuleList(
        (0-31): 32 x MllamaVisionEncoderLayer(
          (self_attn): MllamaVisionSdpaAttention(
            (q_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (k_proj): Linear(in_features=1280, out_features=1280, b

In [5]:
processor = AutoProcessor.from_pretrained(model_id)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [92]:
# url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/0052a70beed5bf71b92610a43a52df6d286cd5f3/diffusers/rabbit.jpg"
#url = "https://distill.pub/2018/building-blocks/examples/input_images/dog_cat.jpeg"
url = "https://hips.hearstapps.com/hmg-prod/images/dog-puppy-on-garden-royalty-free-image-1586966191.jpg"
image = Image.open(requests.get(url, stream=True).raw)

In [93]:
image.show()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [94]:

prompt = "<|image|><|begin_of_text|>this is a picture of a"
inputs = processor(image, prompt, return_tensors="pt").to(model.device)

In [95]:
# inputs

In [96]:
inputs['input_ids']

tensor([[128000, 128256, 128000,    576,    374,    264,   6945,    315,    264]])

In [97]:
def print_decoded(tokens):
  for i, t in enumerate(tokens):
    print(f"{i} Token {t}: \t\t{tokenizer.decode(t)}")

print_decoded(inputs['input_ids'][0])

0 Token 128000: 		<|begin_of_text|>
1 Token 128256: 		<|image|>
2 Token 128000: 		<|begin_of_text|>
3 Token 576: 		this
4 Token 374: 		 is
5 Token 264: 		 a
6 Token 6945: 		 picture
7 Token 315: 		 of
8 Token 264: 		 a


In [98]:
output = model.generate(**inputs, max_new_tokens=5)


In [99]:
print_decoded(output[0])


0 Token 128000: 		<|begin_of_text|>
1 Token 128256: 		<|image|>
2 Token 128000: 		<|begin_of_text|>
3 Token 576: 		this
4 Token 374: 		 is
5 Token 264: 		 a
6 Token 6945: 		 picture
7 Token 315: 		 of
8 Token 264: 		 a
9 Token 5679: 		 dog
10 Token 11961: 		 sitting
11 Token 389: 		 on
12 Token 264: 		 a
13 Token 16763: 		 grass


In [125]:
# To get hidden states for the input, we need to run the model again with the generated sequence
hidden_output = model(**inputs, output_hidden_states=True)

# Access hidden states
hidden_states = hidden_output.hidden_states

In [101]:

print("input tokens", len(inputs['input_ids'][0]))
print("hidden states (layers)", len(hidden_states))
print("hidden state shape", hidden_states[20].shape)

input tokens 9
hidden states (layers) 41
hidden state shape torch.Size([1, 9, 4096])


In [102]:
model.language_model.model.layers

ModuleList(
  (0-2): 3 x MllamaSelfAttentionDecoderLayer(
    (self_attn): MllamaTextSelfSdpaAttention(
      (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
      (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
      (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    )
    (mlp): MllamaTextMLP(
      (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
      (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
      (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
      (act_fn): SiLU()
    )
    (input_layernorm): MllamaTextRMSNorm((4096,), eps=1e-05)
    (post_attention_layernorm): MllamaTextRMSNorm((4096,), eps=1e-05)
  )
  (3): MllamaCrossAttentionDecoderLayer(
    (cross_attn): MllamaTextCrossSdpaAttention(
      (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (k_proj): Linear(in

In [103]:
def gather_residual_activations(model, target_layer, inputs):
  """
  This function allows us to gather activations for a specific layer on a model.
  
  Args:
  - model: The model from which we want to gather activations.
  - target_layer: The specific layer index for which we want to gather activations.
  - inputs: The input data to be passed through the model.
  
  Returns:
  - target_act: The activations of the specified layer.
  """
  target_act = None
  def gather_target_act_hook(mod, inputs, outputs):
    nonlocal target_act # make sure we can modify the target_act from the outer scope
    target_act = outputs[0]
    return outputs
  # we could also easily target the MLP layer
  # handle = model.model.layers[target_layer].mlp.register_forward_hook(gather_mlp_output_hook)
  handle = model.language_model.model.layers[target_layer].register_forward_hook(gather_target_act_hook)
  _ = model.forward(inputs)
  handle.remove()
  return target_act

In [104]:
# these layers don't correspond to the llama 3.1 layers directly... 
target_act = gather_residual_activations(model, 29, inputs['input_ids'])


In [105]:
target_act.shape

torch.Size([1, 9, 4096])

In [106]:
target_act[0].shape

torch.Size([9, 4096])

In [107]:
target_act[0][8]

tensor([ 0.5600, -0.4416,  0.1112,  ...,  0.1245,  0.2329, -0.4605])

In [108]:
hidden_states[29][0][8]

tensor([ 0.2810, -0.3050, -0.0204,  ...,  0.4391, -0.0121,  0.1785])

In [109]:
from latentsae import Sae

In [29]:
# sae = Sae.load_from_hub("EleutherAI/sae-llama-3-8b-32x", hookpoint="layers.10")
sae = Sae.load_from_hub("EleutherAI/sae-llama-3.1-8b-64x", "layers.29")

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 57065.36it/s]


In [27]:
import requests
import json

# URL of the JSON file
url = "https://huggingface.co/datasets/EleutherAI/auto_interp_explanations/resolve/main/Llama/262k/res/model.layers.29_feature.json"

# Fetch the JSON data from the URL
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON data
    feature_data = json.loads(response.text)
    print("JSON data loaded successfully.")
else:
    print(f"Failed to load JSON data. Status code: {response.status_code}")



JSON data loaded successfully.


In [36]:
feature_data["0"]

'Prepositions and conjunctions used to connect entities, objects, locations, and actions, often used to establish relationships between them.'

In [110]:
# sae_latents = sae.encode(target_act)
sae_latents = sae.encode(hidden_states[29])


In [111]:
sae_latents.top_indices.shape

torch.Size([1, 9, 32])

In [112]:
sae_latents.top_indices[0][8]

tensor([247099,  10841,  20496, 191322,   7054, 233130, 228488, 139664,  67798,
        190614, 154686, 243108, 129717,  87905, 237260,  90743, 180116, 222360,
         74101,  73845, 155172, 227097, 204738,  90707, 248969, 107166, 158427,
         92896,  94477, 156573, 212873,  25009])

In [126]:
def show_features(indices, values):
  for i,idx in enumerate(indices):
    try:
      print(f"Feature {idx.item()} ({values[i].item()}): {feature_data[str(idx.item())]}")
    except:
      print(f"Feature {idx.item()}: {values[i].item()}")

token = 6
k = 10
show_features(sae_latents.top_indices[0][token][:k], sae_latents.top_acts[0][token][:k])


Feature 140756: 12.42545223236084
Feature 143465 (6.320024013519287): The word \picture\ is often used in the context of visual representation, including photographs, images, or movies, and can be used as a noun or part of a compound noun.
Feature 7054 (4.330352783203125): Punctuation marks and special characters that separate or connect pieces of information, often used for addresses, phone numbers, dates, citations, and mathematical or scientific notation.
Feature 94477 (3.9756956100463867): Prepositions or conjunctions in common English phrases, sometimes introducing clauses or phrases that provide additional information, often found in formal or technical writing.
Feature 190614 (3.0191426277160645): A single letter or short sequence of letters, sometimes including a name or initials, that is part of a larger name or word.
Feature 255322 (2.8928561210632324): Nouns related to visual or static media, such as images, photographs, videos, or other forms of representation.
Feature 6483